# 03 — Video Topic Generation

Select one focused category from the knowledge tree and generate a structured
set of educational short-video ideas.

This notebook:

1. Loads the saved knowledge tree.
2. Selects a category within a configured depth range.
3. Generates distinct topic ideas with the local LLM.
4. Saves the validated topics as JSON.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from educational_shorts.prompts import load_prompt
from educational_shorts.topics import (
    build_topics_filename,
    generate_topics,
    sample_node_path,
    save_topics,
)
from educational_shorts.tree import load_tree

print(f"Project root: {PROJECT_ROOT}")

Project root: c:\Users\hitch\python_files\educational_shorts


## Configuration

In [2]:
ROOT_CATEGORY = "Science"

TOPIC_COUNT = 10
MIN_CATEGORY_DEPTH = 2
MAX_CATEGORY_DEPTH = 3

CATEGORY_SELECTION_SEED = 42
TOPIC_GENERATION_SEED = 42
TEMPERATURE = 0.7

TREE_PATH = (
    PROJECT_ROOT
    / "data"
    / "knowledge_tree"
    / f"{ROOT_CATEGORY.lower().replace(' ', '_')}.json"
)

TOPICS_DIRECTORY = PROJECT_ROOT / "data" / "topics"

print(f"Tree path: {TREE_PATH}")
print(f"Topics directory: {TOPICS_DIRECTORY}")

Tree path: c:\Users\hitch\python_files\educational_shorts\data\knowledge_tree\science.json
Topics directory: c:\Users\hitch\python_files\educational_shorts\data\topics


## Load the knowledge tree and generation prompt

In [3]:
if not TREE_PATH.exists():
    raise FileNotFoundError(
        f"Knowledge tree not found at {TREE_PATH}. "
        "Run Notebook 02 before generating topics."
    )

knowledge_tree = load_tree(TREE_PATH)
topic_system_prompt = load_prompt("video_topic_generation")

print(f"Loaded tree: {knowledge_tree.name}")
print("Topic-generation prompt loaded.")

Loaded tree: Science
Topic-generation prompt loaded.


## Select a category

In [4]:
selected_category_path = sample_node_path(
    tree=knowledge_tree,
    min_depth=MIN_CATEGORY_DEPTH,
    max_depth=MAX_CATEGORY_DEPTH,
    seed=CATEGORY_SELECTION_SEED,
)

print("Selected category:")
print(" > ".join(selected_category_path))

Selected category:
Science > Biology > Microbiology


## Generate topics

In [5]:
generated_topics = generate_topics(
    category_path=selected_category_path,
    system_prompt=topic_system_prompt,
    count=TOPIC_COUNT,
    temperature=TEMPERATURE,
    seed=TOPIC_GENERATION_SEED,
)

print(f"Generated {len(generated_topics.topics)} topics.")

Generated 10 topics.


## Save topics

In [6]:
output_path = (
    TOPICS_DIRECTORY
    / build_topics_filename(selected_category_path)
)

save_topics(
    topics=generated_topics,
    output_path=output_path,
)

print(f"Saved topics to {output_path}")

Saved topics to c:\Users\hitch\python_files\educational_shorts\data\topics\science__biology__microbiology.json


## Preview

In [7]:
for index, topic in enumerate(generated_topics.topics, start=1):
    print(f"{index}. {topic.title}")
    print(f"   Objective: {topic.learning_objective}")
    print()

1. How Do Bacteria Communicate?
   Objective: Understand how bacteria use chemical signals to communicate and coordinate behavior.

2. What Is a Biofilm, and Why Does It Matter?
   Objective: Learn what biofilms are and their significance in health, industry, and the environment.

3. How Do Antimicrobial Resistance Genes Spread?
   Objective: Explore how bacteria share genes that make them resistant to antibiotics through horizontal gene transfer.

4. What Are the Differences Between Viruses and Bacteria?
   Objective: Compare and contrast viruses and bacteria in terms of structure, reproduction, and impact on health.

5. How Do Microbes Help Break Down Plastic?
   Objective: Discover how certain microbes can degrade plastic waste through specialized enzymes.

6. What Is the Role of Endospores in Bacterial Survival?
   Objective: Understand how endospores allow bacteria to survive extreme environmental conditions.

7. How Do Microbes Influence Human Digestion?
   Objective: Explore the